# Помесячные версии скоринговой модели

Оффер клиенту выбирает скоринговая модель — детерминированно, лучший по скору. Раз в месяц модель
переобучают, и каждая версия работает свой месяц. Вероятности выбора не логировались, версии модели
сохранены. Данные — панель «клиент × месяц»: один клиент есть в нескольких месяцах, в каких-то его нет.

Вопросы:

- какая версия лучшая, если сравнить их на одних и тех же клиентах;
- какой финансовый эффект даёт скоринг целиком по сравнению со случайной выдачей офферов;
- что делать, если оффер иногда меняют менеджеры, и как проверить, не портит ли это оценку;
- можно ли по этим логам оценить новую модель;
- что изменить в проде, чтобы такие вопросы решались точно.

1. Данные: панель «клиент × месяц»
2. Почему одного месяца мало
3. Все месяцы вместе
4. Лучшая версия
5. Эффект скоринга целиком
6. Если оффер меняют менеджеры
7. Новая модель
8. Случайный срез в проде
9. Сезонность
10. Чек-лист

Данные синтетические, поэтому каждый результат сверяется с истиной.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits

import policyscope as ps

SEED = 4             # зерно симулятора
MONTHS = 12          # месяцев (версий модели)
CLIENTS = 3_000      # клиентов в панели
PRESENCE = 0.8       # доля месяцев, в которых клиент есть в логах
DRIFT = 0.35         # насколько версия меняется за одно переобучение
EXPLORATION = 0.10   # доля случайных показов в разделе 8
N_BOOT = 100         # bootstrap-реплик там, где пропенсити оцениваются

pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", " ") if abs(x) >= 1e4 else f"{x:.4g}")
warnings.filterwarnings("ignore", message="Bootstrap:")
# Один поток BLAS/OpenMP: на части машин (например, в WSL) многопоточный бустинг sklearn
# в десятки раз медленнее однопоточного. Библиотека внутри и так обучает модели в один поток.
threadpool_limits(1)


def values_vs_truth(board, policies, on, metric):
    """V каждого кандидата рядом с истиной на тех же строках (по одной метрике)."""
    rows = []
    for name, rep in board.reports.items():
        m = rep[metric]
        t = sim.true_value(on, policies[name], metric)
        rows.append({"кандидат": name, "оценка V": m.v_b.value, "ДИ от": m.v_b.low, "ДИ до": m.v_b.high,
                     "истина": t, "ДИ накрывает истину": "да" if m.v_b.low <= t <= m.v_b.high else "нет"})
    return pd.DataFrame(rows).set_index("кандидат")


def as_table(shares, frame):
    """Случайная политика с долями ``shares`` как таблица — для расчёта истины."""
    return pd.DataFrame(np.tile(shares.reindex(sim.actions, fill_value=0).to_numpy(), (len(frame), 1)),
                        index=frame.index, columns=sim.actions)


def warnings_of(report):
    """Заголовок вывода и находки, которые снижают доверие."""
    icons = {"block": "⛔", "caution": "⚠️"}
    for metric, m in report.metrics.items():
        print(f"{metric}: {m.sentence(report.policy, report.baseline)}")
        for f in m.findings:
            if f.severity in icons:
                print(f"   {icons[f.severity]} {f.title}")

## 1. Данные: панель «клиент × месяц»

Ваш формат: `id, report_date, [признаки], [отклики], A_offer` — строка на клиента в месяце, дата —
конец месяца. Разметка колонок задаётся явно, без авто-выбора:

- `action="A_offer"` — что показала текущая политика;
- `reward` — отклики, `features` — признаки клиента на дату решения;
- `cluster="id"` — строки одного клиента зависимы: интервалы и bootstrap учитывают это, а фолды
  моделей не смешивают месяцы одного клиента.

Для сравнения версий нужны и **все версии модели**: библиотека пересчитает, что каждая версия
рекомендовала бы любому клиенту. В симуляторе их даёт `sim.versions` — каждая следующая версия —
предыдущая с небольшими изменениями, как при ежемесячном переобучении.

In [2]:
sim = ps.synthetic.OfferSimulator(seed=SEED, client_effect=1.0)
dates = [str(pd.Period(f"2025-{m:02d}", "M").end_time.date()) for m in range(1, MONTHS + 1)]
versions = sim.versions(MONTHS, drift=DRIFT, seed=1, names=dates)
base = sim.sample_clients(CLIENTS)
presence = np.random.default_rng(SEED).random((CLIENTS, MONTHS)) < PRESENCE


def make_panel(choose, shift=None):
    """Логи «клиент × месяц»: choose(date, clients) — вероятности офферов, shift — сезонный сдвиг отклика."""
    frames = []
    for j, date in enumerate(dates):
        present = base[presence[:, j]]
        month = sim.simulate(present, choose(date, present), shift=(shift or {}).get(date, 0.0))
        frames.append(month.assign(report_date=date))
    panel = pd.concat(frames, ignore_index=True).drop(columns="propensity")
    panel = panel.rename(columns={"client_id": "id", "offer": "A_offer"})
    return panel.sort_values(["id", "report_date"], ignore_index=True)


def by_version(date, clients):
    return versions[date].action_probs(clients)


panel = make_panel(by_version)
FEATURES = ["age", "income", "loyal", "risk", "region"]
COLUMNS = dict(action="A_offer", reward=["accepted", "revenue"], features=FEATURES, cluster="id")
print(f"{len(panel)} строк, {panel['id'].nunique()} клиентов, месяцев на клиента: "
      f"от {panel.groupby('id').size().min()} до {panel.groupby('id').size().max()}")
panel.head()

28774 строк, 3000 клиентов, месяцев на клиента: от 5 до 12


,id,age,income,loyal,risk,region,A_offer,accepted,revenue,report_date
0,0,69,109 700,1,0.33,center,card,0,0,2025-02-28
1,0,69,109 700,1,0.33,center,card,0,0,2025-04-30
2,0,69,109 700,1,0.33,center,card,0,0,2025-05-31
3,0,69,109 700,1,0.33,center,card,0,0,2025-06-30
4,0,69,109 700,1,0.33,center,card,0,0,2025-08-31


Насколько версии расходятся: сколько разных офферов клиенту рекомендовали бы 12 версий.

In [3]:
recommended = pd.DataFrame({d: versions[d].action_probs(base).idxmax(axis=1) for d in dates})
recommended.nunique(axis=1).value_counts(normalize=True).sort_index().rename_axis(
    "разных офферов от всех версий").to_frame("доля клиентов")

,доля клиентов
разных офферов от всех версий,
1,0.4403
2,0.431
3,0.1107
4,0.018


Для клиентов, которым все версии рекомендуют одно и то же, логи знают отклик только на этот оффер.
Для остальных было исследование: оффер зависел от того, в каком месяце клиент пришёл.

## 2. Почему одного месяца мало

Одна версия детерминирована: каждому клиенту — один оффер. Если другая политика выбирает иначе, об
отклике на её оффер у этого клиента логи месяца ничего не знают. Такую часть оценка лишь
экстраполирует моделью исхода, а интервал ошибку экстраполяции не учитывает. Отчёт знает эту долю
точно: больше 5% — вывод «с осторожностью», больше 20% — вывод не делается.

In [4]:
one_month = panel[panel["report_date"] == dates[0]]
for name, candidate in {"следующая версия": versions[dates[1]], "случайная выдача": ps.random_like()}.items():
    report = ps.evaluate(one_month, candidate, **COLUMNS, logging_policy=versions[dates[0]], name=name,
                         inference="none")
    print(f"— {name}: решений вне покрытия месяца {report.overlap.uncovered:.0%}")
    warnings_of(report)

— следующая версия: решений вне покрытия месяца 11%
accepted: Δ = −0.01 п.п. (−0.0%); доверительный интервал не считался.
   ⚠️ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся
revenue: Δ = −1.44 (−3.3%); доверительный интервал не считался.
   ⚠️ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся


— случайная выдача: решений вне покрытия месяца 69%
accepted: Δ = −6.28 п.п. (−17.8%); доверительный интервал не считался.
   ⛔ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся
revenue: Δ = +0.108 (+0.2%); доверительный интервал не считался.
   ⛔ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся


Близкую версию по одному месяцу можно оценить лишь с осторожностью, случайную выдачу — никак.

## 3. Все месяцы вместе

Передаём словарь версий и колонку месяца. Вероятность показать клиенту оффер — доля месяцев, версии
которых его рекомендуют (смесь версий, balanced IPS). Пропенсити известны точно, поэтому интервал
аналитический, а модель пропенсити не нужна.

Предпосылка — сопоставимые месяцы. Библиотека проверяет две вещи:

- **состав клиентов** — угадывается ли месяц по признакам клиента;
- **отклик** — на клиентах, которым все версии рекомендуют одно и то же, он не должен зависеть от
  месяца (иначе это сезонность).

In [5]:
ev = ps.Evaluator(panel, **COLUMNS, logging_policy=versions, logger="report_date")
check = ev.evaluate(versions[dates[-1]], name=dates[-1])
print(next(f.detail for f in check.findings if f.code == "logging_mixture"))

Логи записаны 12 политиками (колонка «report_date»: 2025-01-31 — 8%, 2025-02-28 — 8%, 2025-03-31 — 8%, 2025-04-30 — 8%, 2025-05-31 — 8%, 2025-06-30 — 8%, 2025-07-31 — 8%, 2025-08-31 — 8%, 2025-09-30 — 8%, 2025-10-31 — 8%, 2025-11-30 — 8%, 2025-12-31 — 8%). Оценка несмещена, если клиенты и их отклик в периодах разных политик сопоставимы. Период по признакам клиента угадывается с ROC AUC 0.50: сдвиг состава клиентов не обнаружен. Сезонность проверена на 12603 строках, где все политики выбирают одно и то же: различий отклика между периодами не обнаружено.


## 4. Лучшая версия

Все версии оцениваются на всех клиентах сразу. База **A** — наблюдаемая политика: каждый месяц
работала своя версия. `best` выбирает лидера:

- набор лучших — допущенные версии, которых ни одна допущенная значимо не обходит; если в нём больше
  одной версии, выбрать между ними по этим логам нельзя;
- одновременный интервал лидера — честный, хотя лидер выбран по тем же данным;
- ограничитель — допускаются только версии, для которых доказано, что доля принятых офферов падает
  относительно A не больше чем на 0.5 п.п. (при малой мощности не допускается и хорошая версия).

In [6]:
board = ev.compare(versions)
selection = board.best("revenue", guardrails={"accepted": 0.005})
selection

- Лидер — «2025-12-31»: Δ к A = +10.07, одновременный 95% ДИ [+7.75; +12.40]. Этот интервал честен и после выбора лидера; точечная оценка лидера завышена (winner's curse).
- Лидер значимо лучше остальных допущенных кандидатов.
- Не допущен «2025-04-30»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-03-31»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-01-31»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-02-28»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.

Сверка с истиной. Для сравнения — «наивная» оценка: средняя выручка в месяце, когда версия работала.
Она видит версию только на клиентах одного месяца, поэтому её интервал заметно шире.

In [7]:
table = values_vs_truth(board, versions, on=panel, metric="revenue")
own = panel.groupby("report_date")["revenue"].agg(["mean", "std", "count"])
table["наивно: среднее за свой месяц"] = own["mean"]
table["ширина ДИ: OPE"] = table["ДИ до"] - table["ДИ от"]
table["ширина ДИ: наивно"] = 2 * 1.96 * own["std"] / np.sqrt(own["count"])
table.sort_values("истина", ascending=False).head(6)

,оценка V,ДИ от,ДИ до,истина,ДИ накрывает истину,наивно: среднее за свой месяц,ширина ДИ: OPE,ширина ДИ: наивно
кандидат,,,,,,,,
2025-12-31,61.91,59.46,64.36,60.76,да,62.2,4.901,7.268
2025-06-30,58.12,55.99,60.25,57.4,да,56.81,4.263,6.648
2025-10-31,55.74,53.78,57.69,55.86,да,53.21,3.911,6.249
2025-11-30,56.2,54.19,58.21,55.83,да,60.31,4.027,6.648
2025-09-30,54.95,52.9,57,54.49,да,55.22,4.103,6.662
2025-05-31,53.59,51.63,55.55,53.16,да,55.08,3.912,6.495


## 5. Эффект скоринга целиком

Бизнес-вопрос: сколько денег приносит скоринг по сравнению со случайной выдачей офферов. Честная база
— случайная выдача **с тем же балансом офферов**, что у A: `ps.random_like()`. Тогда разница — вклад
именно выбора «кому что».

Почему не помесячный баланс (`ps.random_like(by="report_date")`). При сопоставимых месяцах — а это
предпосылка смеси версий — месяц не связан с клиентом, и у случайной выдачи с помесячным и с общим
балансом одно и то же значение. Но для политики, зависящей от месяца, оценка по смеси версий хуже: её
вероятности выше в тех месяцах, чья версия рекомендует тот же оффер, IPS смещается по построению, а DR
сильнее опирается на модель исхода (отчёт отметит это находкой). Помесячный баланс уместен, когда
пропенсити свои у каждой строки: колонка пропенсити или одна логирующая политика.

Прямо оценить случайную выдачу по этим логам нельзя:

In [8]:
random_policy = ps.random_like()
warnings_of(ev.evaluate(random_policy, name="случайная"))

accepted: вывод сделать нельзя: оценка Δ = −8.30 п.п. (−21.7%) ненадёжна (причины ниже).
   ⛔ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся
revenue: вывод сделать нельзя: оценка Δ = −6.09 (−11.7%) ненадёжна (причины ниже).
   ⛔ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся


Случайная выдача назначает клиентам офферы, которых им не рекомендовала ни одна версия, — откликов на
них в логах нет. Но кое-что сказать можно. `ev.bounds` делит решения случайной политики на две части:

- **покрытая** — оценивается как обычно: DR с точными пропенсити версий;
- **непокрытая** — отклик на неё неизвестен, но лежит в пределах метрики: доля — от 0 до 1, выручка —
  от 0 до максимума.

Отсюда границы без допущений об отклике на непоказанные офферы и таблица чувствительности: что будет,
если этот отклик равен λ × прогноз модели исхода. Покрытая часть опирается на те же предпосылки, что
и раздел 3: точные пропенсити версий и сопоставимые месяцы. Верхняя граница выручки по умолчанию —
наблюдаемый максимум одной покупки. Для *среднего* отклика это щедрая граница, но формально допущение,
и отчёт его помечает; на своих данных задавайте её из бизнес-смысла: `high={"revenue": ...}`.

In [9]:
effect = ev.bounds(random_policy, name="случайная")
effect

## Границы: случайная против A

46.9% решений «случайная» приходятся на действия, которых логирующая политика этим клиентам не показывала: отклик на них в логах не наблюдался. Ниже — что можно сказать о Δ = V(случайная) − V(A) без допущений об этом отклике и с ними.

| Метрика | Диапазон отклика | V(случайная): границы | Δ: границы | 95% ДИ для Δ | Δ не больше (95%) | Δ не меньше (95%) |
|---|---|---|---|---|---|---|
| accepted | [0; 1] (бинарная) | [19.39%; 66.26%] | [−18.79 п.п.; +28.08 п.п.] | [−19.35 п.п.; +28.69 п.п.] | +28.69 п.п. | −19.35 п.п. |
| revenue | [0; 779.95] (low — наблюдаемый min, high — наблюдаемый max) | [25.76; 391.3] | [−26.07; +339.5] | [−27.05; +344.2] | +344.2 | −27.05 |

Границы верны при любом отклике на непоказанные действия в пределах диапазона. «Не больше» и «не меньше» — каждая отдельная односторонняя 95%-граница; интервал накрывает истинное Δ с вероятностью не меньше 95% (Imbens–Manski). Покрытая часть оценивается как обычно и опирается на точные пропенсити заданной логирующей политики и на сопоставимость периодов (проверки ниже).
Для revenue диапазон хотя бы с одной стороны взят по наблюдаемым значениям — это допущение: средний отклик на непоказанные действия не выходит за наблюдённые. Задайте его явно (low=, high=).

- accepted: V(A) − V(случайная) не больше +19.35 п.п. и, отдельно, не меньше −28.69 п.п. (односторонние 95%); вместе с вероятностью 95% — в [−28.69 п.п.; +19.35 п.п.].
- revenue: V(A) − V(случайная) не больше +27.05 и, отдельно, не меньше −344.2 (односторонние 95%); вместе с вероятностью 95% — в [−344.2; +27.05].

### Чувствительность

Если отклик на непоказанные действия равен λ × прогноз модели исхода (λ = 1 — модель верна, λ > 1 — эти клиенты откликались бы охотнее, чем она предсказывает):

| Метрика | λ = 0.5 | λ = 0.75 | λ = 1 | λ = 1.25 | λ = 1.5 | Δ меняет знак при λ |
|---|---|---|---|---|---|---|
| accepted | −13.55 п.п. | −10.93 п.п. | −8.30 п.п. | −5.70 п.п. | −3.15 п.п. | 1.82 |
| revenue | −16.08 | −11.09 | −6.09 | −1.23 | +3.38 | 1.32 |

### Предпосылки покрытой части
- ℹ️ **Пропенсити посчитаны по смеси логирующих политик**. Логи записаны 12 политиками (колонка «report_date»: 2025-01-31 — 8%, 2025-02-28 — 8%, 2025-03-31 — 8%, 2025-04-30 — 8%, 2025-05-31 — 8%, 2025-06-30 — 8%, 2025-07-31 — 8%, 2025-08-31 — 8%, 2025-09-30 — 8%, 2025-10-31 — 8%, 2025-11-30 — 8%, 2025-12-31 — 8%). Оценка несмещена, если клиенты и их отклик в периодах разных политик сопоставимы. Период по признакам клиента угадывается с ROC AUC 0.50: сдвиг состава клиентов не обнаружен. Сезонность проверена на 12603 строках, где все политики выбирают одно и то же: различий отклика между периодами не обнаружено.

In [10]:
truth_A = {m: sum(sim.true_value(panel[panel["report_date"] == d], versions[d], m) * (panel["report_date"] == d).sum()
                  for d in dates) / len(panel) for m in ("accepted", "revenue")}
random_table = as_table(panel["A_offer"].value_counts(normalize=True), panel)
pd.DataFrame({
    m: {"истинный эффект скоринга V(A) − V(случайная)": truth_A[m] - sim.true_value(panel, random_table, m),
        "не больше (95%)": -effect[m].at_least,
        "оценка при λ = 1": -effect[m].sensitivity[1.0],
        "вывод «скоринг лучше» держится при λ до": effect[m].breakdown}
    for m in ("accepted", "revenue")
})

,accepted,revenue
истинный эффект скоринга V(A) − V(случайная),0.07845,6.789
не больше (95%),0.1935,27.05
оценка при λ = 1,0.08303,6.09
вывод «скоринг лучше» держится при λ до,1.819,1.316


Что из этого следует:

- **«Не больше» — без допущений об отклике на непоказанные офферы.** Эффект скоринга не превышает этой
  величины с вероятностью 95%, как бы клиенты ни откликались на непоказанные им офферы (в пределах
  диапазона метрики).
- **«Не меньше» без допущений не бывает.** Если бы непоказанные офферы нравились клиентам максимально,
  случайная выдача могла бы оказаться лучше. Нижнюю оценку даёт только допущение, и таблица
  чувствительности формулирует его на языке бизнеса: вывод «скоринг лучше» сохраняется, пока отклик
  на непоказанные офферы не выше прогноза модели в λ* раз (последняя строка таблицы).
- **Точный ответ даёт контрольная группа**: небольшой доле клиентов на месяц-два — случайный оффер. Нужный
  объём прикидывается заранее: $n \approx (2.8 \cdot SD / \Delta)^2$ решений в контрольной группе для
  мощности 80%, где Δ — ожидаемый эффект.

In [11]:
for m in ("accepted", "revenue"):
    planned = abs(effect[m].sensitivity[1.0])
    n_control = (2.8 * panel[m].std() / planned) ** 2
    print(f"{m}: при эффекте около {planned:.3g} контрольной группе нужно ~{n_control:,.0f} решений".replace(",", " "))

accepted: при эффекте около 0.083 контрольной группе нужно ~268 решений
revenue: при эффекте около 6.09 контрольной группе нужно ~1 291 решений


## 6. Если оффер меняют менеджеры

Пусть скоринг рекомендует оффер (`A_rec`), а менеджер в части случаев показывает другой (`A_offer`).
Тогда наблюдаемая политика — уже не версии модели, и `logging_policy=versions` справедливо откажется
работать: залогированный оффер версия выбрать не могла.

In [12]:
DEVIATION = 0.25


def with_managers(informed):
    """Менеджер меняет рекомендацию: случайно или когда видит, что клиент охотнее откликнется."""
    def choose(date, clients):
        rec = versions[date].action_probs(clients).to_numpy()
        if informed:  # решение зависит от скрытой склонности клиента, которой нет в признаках
            d = 1 / (1 + np.exp(-(2.0 * sim.hidden_trait(clients) - 1.2)))
        else:
            d = np.full(len(clients), DEVIATION)
        probs = (1 - d)[:, None] * rec + d[:, None] * (1 - rec) / 3
        return pd.DataFrame(probs, index=clients.index, columns=sim.actions)
    logs = make_panel(choose)
    logs["A_rec"] = pd.concat([versions[d].action_probs(g).idxmax(axis=1)
                               for d, g in logs.groupby("report_date")]).sort_index()
    logs["accepted_prev"] = logs.groupby("id")["accepted"].shift(1).fillna(logs["accepted"].mean())
    return logs


random_managers = with_managers(informed=False)
print(f"Менеджеры изменили оффер в {np.mean(random_managers['A_offer'] != random_managers['A_rec']):.0%} случаев")
try:
    ps.Evaluator(random_managers, **COLUMNS, logging_policy=versions, logger="report_date")
except ValueError as error:
    print("Ошибка:", error)

Менеджеры изменили оффер в 25% случаев


Ошибка: В части строк политика их периода не могла выбрать залогированное действие (вероятность 0); число таких строк по периодам: «2025-02-28» — 599, «2025-04-30» — 596, «2025-05-31» — 606, «2025-06-30» — 579, «2025-08-31» — 600. Значит, задана не та политика, что работала, или в логах есть строки из другого источника. Если рекомендацию иногда меняют менеджеры или каналы, логирующая политика не совпадает с моделью: не задавайте logging_policy, а оцените пропенсити по данным, добавив рекомендацию модели в признаки.


Зато отклонения менеджеров — это исследование: клиенту иногда показывают не то, что рекомендует
модель. Использовать его можно, только если отклонения **не зависят от того, как клиент
отреагирует**. Иначе это скрытый фактор, и оценка смещена без предупреждения.

Как это проверить: логировать рекомендацию модели рядом с фактическим оффером и дать модели
пропенсити рекомендацию признаком. Тогда она выучит, где и как часто менеджеры отклоняются. А
**плацебо-проверка** на отклике прошлого месяца (`placebo="accepted_prev"`) ловит отклонения «со
знанием»: решение этого месяца не может изменить прошлый отклик, поэтому «эффект» на нём должен быть
нулевым.

In [13]:
MANAGER_COLUMNS = dict(COLUMNS, features=FEATURES + ["A_rec"])


def scoring_effect(logs, inference="auto", logging_table=None):
    """Эффект случайной выдачи против A и плацебо; ``logging_table`` — известная политика A для истины."""
    report = ps.evaluate(logs, random_policy, **MANAGER_COLUMNS, placebo="accepted_prev", name="случайная",
                         n_boot=N_BOOT, inference=inference)
    value_a = logs["revenue"].mean() if logging_table is None else sim.true_value(logs, logging_table, "revenue")
    truth = sim.true_value(logs, as_table(logs["A_offer"].value_counts(normalize=True), logs), "revenue") - value_a
    m = report["revenue"]
    interval = f" [{m.delta.low:+.3g}; {m.delta.high:+.3g}]" if m.delta.low is not None else ""
    print(f"Δ выручки, случайная − A: {m.delta.value:+.3g}{interval}; истина ≈ {truth:+.3g}")
    check = report.placebo[0]
    print(f"Плацебо «{check.column}»: Δ = {check.estimate:+.3g} [{check.low:+.3g}; {check.high:+.3g}] — "
          + ("пройдена" if check.status == "ok" else "НЕ пройдена"))
    warnings_of(report)


print("— менеджеры меняют оффер случайно")
rec = pd.get_dummies(random_managers["A_rec"]).reindex(columns=sim.actions, fill_value=0).astype(float)
scoring_effect(random_managers, logging_table=(1 - DEVIATION) * rec + DEVIATION * (1 - rec) / 3)

— менеджеры меняют оффер случайно


Δ выручки, случайная − A: -4.85 [-6.05; -3.58]; истина ≈ -4.44
Плацебо «accepted_prev»: Δ = +0.000432 [-0.00769; +0.00856] — пройдена
accepted: случайная хуже A: Δ = −5.31 п.п. (−15.2%), ДИ [−5.96 п.п.; −4.56 п.п.].
   ⚠️ Пропенсити оценены моделью
revenue: случайная хуже A: Δ = −4.85 (−9.3%), ДИ [−6.05; −3.58].
   ⚠️ Пропенсити оценены моделью


Отклонения случайны — плацебо пройдена, и эффект скоринга оценивается целиком: случайность менеджеров
дала данные по всем офферам. Теперь пусть менеджеры отклоняются со знанием клиента:

In [14]:
print("— менеджеры меняют оффер, когда видят, что клиент охотнее откликнется")
scoring_effect(with_managers(informed=True), inference="none")  # интервал здесь не нужен: смотрим на плацебо

— менеджеры меняют оффер, когда видят, что клиент охотнее откликнется


Δ выручки, случайная − A: +6.83; истина ≈ -5.91
Плацебо «accepted_prev»: Δ = +0.0596 [+0.052; +0.0671] — НЕ пройдена
accepted: Δ = +3.30 п.п. (+9.7%); доверительный интервал не считался.
   ⚠️ Пропенсити оценены моделью
   ⚠️ Плацебо-проверка не пройдена
revenue: Δ = +6.83 (+12.3%); доверительный интервал не считался.
   ⚠️ Пропенсити оценены моделью
   ⚠️ Плацебо-проверка не пройдена


Плацебо не пройдена: клиенты, которым менеджеры меняли оффер, отличались ещё до решения. Такой оценке
верить нельзя — нужна случайность, которую задаёт система, а не люди (раздел 8).

## 7. Новая модель

Команда обучила новую модель отклика. В вашем формате её решение — колонка `B_offer`. По логам версий
новую модель можно оценить только там, где она выбирает офферы, которые клиенту рекомендовала хотя бы
одна версия; эту долю отчёт знает точно.

In [15]:
train = make_panel(by_version)  # логи других месяцев той же системы — на них учится новая модель


def design(df, offer):
    X = pd.get_dummies(df[FEATURES], columns=["region"], dtype=float)
    for a in sim.actions:
        X[f"offer={a}"] = (offer == a).astype(float) if isinstance(offer, pd.Series) else float(offer == a)
    return X


response = HistGradientBoostingClassifier(random_state=0).fit(design(train, train["A_offer"]), train["accepted"])
avg_revenue = train[train["accepted"] == 1].groupby("A_offer")["revenue"].mean()


def new_scores(df):
    """Ожидаемая выручка каждого оффера по новой модели."""
    return pd.DataFrame({a: response.predict_proba(design(df, a))[:, 1] * avg_revenue[a] for a in sim.actions},
                        index=df.index)


# Когда колонка B_offer есть в логах с самого начала, достаточно ev.evaluate("B_offer");
# здесь модель обучена после создания ev, поэтому её решения передаются как Series.
panel["B_offer"] = new_scores(panel).idxmax(axis=1)
new_report = ev.evaluate(panel["B_offer"], name="новая")
print(f"Решения новой модели вне покрытия логов: {new_report.overlap.uncovered:.0%}")
warnings_of(new_report)

Решения новой модели вне покрытия логов: 74%
accepted: вывод сделать нельзя: оценка Δ = −2.53 п.п. (−6.6%) ненадёжна (причины ниже).
   ⛔ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся
revenue: вывод сделать нельзя: оценка Δ = +62.80 (+121.2%) ненадёжна (причины ниже).
   ⛔ Часть решений — вне покрытия логов
   ⚠️ IPS и DR расходятся


Большая часть решений новой модели — офферы, которых версии этим клиентам не давали, и вывод
заблокирован. Корректно оценивается более осторожная политика — **новая модель в пределах версий**:
лучший по её скорингу оффер среди тех, что клиенту уже рекомендовала хоть одна версия
(`allowed=ev.support()`). Она показывает, лучше ли новая модель ранжирует уже используемые варианты, но
не измеряет выигрыш от новых. Для новой модели целиком подойдут и границы: `ev.bounds(panel["B_offer"])`.

In [16]:
inside = ps.greedy(new_scores(panel), allowed=ev.support())
values_vs_truth(ev.compare({"новая в пределах версий": inside}), {"новая в пределах версий": inside},
                on=panel, metric="revenue")

,оценка V,ДИ от,ДИ до,истина,ДИ накрывает истину
кандидат,,,,,
новая в пределах версий,66.72,64.06,69.37,66.46,да


## 8. Случайный срез в проде

Как сделать все эти вопросы решаемыми точно: в каждом месяце небольшая доля показов (здесь 10%) —
случайный оффер, назначенный системой. Логирующая политика месяца — смесь версии и равномерной выдачи;
её вероятности известны точно.

In [17]:
def with_slice(date, clients):
    return (1 - EXPLORATION) * versions[date].action_probs(clients) + EXPLORATION / len(sim.actions)


panel_x = make_panel(with_slice)
panel_x["B_offer"] = new_scores(panel_x).idxmax(axis=1)
logging_x = {d: ps.mixture([versions[d], ps.uniform()], weights=[1 - EXPLORATION, EXPLORATION]) for d in dates}
ev_x = ps.Evaluator(panel_x, **COLUMNS, logging_policy=logging_x, logger="report_date")
best_version = selection.winner
board_x = ev_x.compare({
    f"лучшая версия ({best_version})": versions[best_version],
    "новая модель": "B_offer",
    "случайная": ps.random_like(),
})
board_x

## Сравнение кандидатов с A

| Кандидат | Метрика | V(A) [95% ДИ] | V(кандидата) [95% ДИ] | Δ | 95% ДИ для Δ | Δ, % | p | p (RW) | Вывод | Доверие | Оценщики |
|---|---|---|---|---|---|---|---|---|---|---|---|
| лучшая версия (2025-12-31) | accepted | 37.40% [36.45%; 38.34%] | 40.13% [38.98%; 41.29%] | +2.74 п.п. | [+2.10 п.п.; +3.37 п.п.] | +7.3% | <0.001 | <0.001 | лучше A | надёжно | согласованы |
| лучшая версия (2025-12-31) | revenue | 52.06 [50.41; 53.70] | 61.75 [59.29; 64.20] | +9.69 | [+8.07; +11.31] | +18.6% | <0.001 | <0.001 | лучше A | надёжно | согласованы |
| новая модель | accepted | 37.40% [36.45%; 38.34%] | 36.35% [33.45%; 39.25%] | −1.05 п.п. | [−3.92 п.п.; +1.82 п.п.] | −2.8% | 0.473 | 0.479 | разница не обнаружена | с осторожностью | согласованы |
| новая модель | revenue | 52.06 [50.41; 53.70] | 104.1 [93.82; 114.3] | +52.01 | [+42.03; +61.98] | +99.9% | <0.001 | <0.001 | лучше A | с осторожностью | согласованы |
| случайная | accepted | 37.40% [36.45%; 38.34%] | 30.62% [29.28%; 31.95%] | −6.78 п.п. | [−7.97 п.п.; −5.60 п.п.] | −18.1% | <0.001 | <0.001 | хуже A | надёжно | согласованы |
| случайная | revenue | 52.06 [50.41; 53.70] | 45.44 [43.27; 47.60] | −6.62 | [−8.55; −4.69] | −12.7% | <0.001 | <0.001 | хуже A | надёжно | согласованы |

p (RW) скорректированы методом Romano–Wolf на 6 пар «кандидат × метрика».

### Кто лучший по «accepted»
- Лидер — «лучшая версия (2025-12-31)»: Δ к A = +2.74 п.п., одновременный 95% ДИ [+1.98 п.п.; +3.49 п.п.]. Этот интервал честен и после выбора лидера; точечная оценка лидера завышена (winner's curse).
- Лидер значимо лучше остальных допущенных кандидатов.

Лидер против остальных (одновременные 95% ДИ по всем парам кандидатов, p с поправкой Romano–Wolf; все пары — в `board.pairwise`):

| Против | Разница «лучшая версия (2025-12-31)» − кандидат | 95% ДИ | p (RW) |
|---|---|---|---|
| новая модель | +3.79 п.п. | [+0.27 п.п.; +7.30 п.п.] | 0.013 |
| случайная | +9.52 п.п. | [+7.84 п.п.; +11.19 п.п.] | <0.001 |

Другая метрика выбора или ограничители — `board.best(metric, guardrails={...})`.

In [18]:
values_vs_truth(board_x, on=panel_x, metric="revenue", policies={
    f"лучшая версия ({best_version})": versions[best_version],
    "новая модель": pd.get_dummies(panel_x["B_offer"]).reindex(columns=sim.actions, fill_value=0).astype(float),
    "случайная": as_table(panel_x["A_offer"].value_counts(normalize=True), panel_x),
})

,оценка V,ДИ от,ДИ до,истина,ДИ накрывает истину
кандидат,,,,,
лучшая версия (2025-12-31),61.75,59.29,64.2,60.76,да
новая модель,104.1,93.82,114.3,107,да
случайная,45.44,43.27,47.6,45.8,да


Решения вне покрытия версий опираются только на строки среза, где вес каждой строки велик (здесь до
1 / 2.5% = 40). Если эффективная выборка мала, отчёт снижает доверие.

**Какой срез нужен.** Решения политики вне покрытия версий оцениваются только по строкам среза.
Эффективный размер выборки прикидывается заранее: $ESS \approx n \cdot \varepsilon / (k \cdot u)$,
где $\varepsilon$ — доля среза, $k$ — число офферов, $u$ — доля решений политики вне покрытия версий
(раздел 7). Для надёжной оценки нужны хотя бы тысячи эффективных строк.

In [19]:
u = new_report.overlap.uncovered
planned = len(panel_x) * EXPLORATION / (len(sim.actions) * u)
print(f"Новая модель — прогноз ESS: {planned:,.0f}; по факту: {board_x['новая модель'].overlap.ess:,.0f}".replace(",", " "))

Новая модель — прогноз ESS: 976; по факту: 932


## 9. Сезонность

Месяцы могут различаться не только версией модели. Смоделируем декабрь, в котором клиенты откликаются
заметно охотнее, — отчёт заметит это на клиентах, которым все версии рекомендуют одно и то же.

In [20]:
seasonal = make_panel(by_version, shift={dates[-1]: 0.6})
seasonal_report = ps.evaluate(seasonal, versions[dates[0]], **COLUMNS, logging_policy=versions, logger="report_date")
print(next(f.detail for f in seasonal_report.findings if f.code == "logger_drift"))

На 12603 строках, где все политики выбирают одно и то же, средний отклик «accepted» с поправкой на признаки различается между периодами на 0.0961 (p = 2.2e-05, с учётом повторов клиентов). Похоже на сезонность или изменения продукта: смесь периодов смешивает и эти различия.


В таком случае смесь месяцев смешивает и сезонный эффект: сравнивайте сопоставимые месяцы или
исключите аномальные.

## 10. Чек-лист

1. Сохраняйте каждую версию модели вместе с правилом выбора оффера.
2. Логируйте месяц (версию), рекомендацию модели, фактически показанный оффер и признаки клиента на
   момент решения. Колонки размечайте явно (`features=`), клиента передавайте как `cluster`.
3. Держите небольшой случайный срез, 5–10% показов, назначаемый системой: с ним точно решаются все
   вопросы этого туториала. Без него:
   - версии и их смеси сравниваются точно (`compare`, `best`);
   - эффект скоринга целиком — только границей «не больше» и анализом чувствительности (`bounds`);
   - новая модель — только в пределах уже показанных офферов (`allowed=ev.support()`).
4. Если оффер меняют люди, проверяйте это плацебо на показателе до решения (`placebo=`). Не прошла —
   оценке верить нельзя. Прошла — данные лишь ограничивают силу скрытых факторов, связанных с этим
   показателем: отчёт пишет, какое смещение Δ ещё не исключено.
5. Сравнивайте сопоставимые месяцы и смотрите на находки о составе клиентов и сезонности.
6. Лучшую версию выбирайте через `board.best(...)`. Если в наборе лучших несколько версий, нужны
   свежие данные или A/B-тест.

Общий разбор библиотеки — в [tutorial.ipynb](tutorial.ipynb), методология — в
[docs/methodology.md](../docs/methodology.md).